In [1]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2025 Hammerheads Engineers sp. z o.o.
# Author: Aleksander Stanik
import sys
import os
import time
import yaml
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, '..'))

if repo_root not in sys.path:
    sys.path.append(repo_root)


import spx_python
# Initialize HTTP-based SPX client wrapper pointing to local SPX server
product_key = os.environ['SPX_PRODUCT_KEY']
wrapper = spx_python.init(address='http://localhost:8000',
                                product_key=product_key)

In [2]:
print (spx_python.__version__)

0.1.0-rc.7


In [4]:
interpolated_signal_model_json ='''
{
  "attributes": {
    "signal_value": 0.0
  },
  "actions": [
    {
      "interpolate": "$attr(signal_value)",
      "points": [
        [0, 0],
        [5, 10],
        [10, 2],
        [15, 15]
      ],
      "method": "cubic",
      "fill_value": "extrapolate"
    }
  ]
}
'''

# Parse JSON and build the model
data = json.loads(interpolated_signal_model_json)

wrapper["models"]["interpolated_model"] = data
wrapper["instances"].clear()
wrapper["instances"]["test_interpolated_model"] = "interpolated_model"

instance = wrapper["instances"]["test_interpolated_model"]

instance["actions"]["interpolate"].points = [
    [0, 0],
    [5, 30],
    [10, 2],
    [15, 15]
]

print("Available models:", wrapper["models"].keys())
print("Available instances:", wrapper["instances"].keys())

Available models: ['TemperatureSensor', 'PowerSupply', 'PIDController', 'interpolated_model']
Available instances: ['test_interpolated_model']


In [5]:
# Define the simulation time range
times = np.linspace(-5, 20, 1000)  # From -5 to 20 seconds

# Initialize lists to store simulation data
signal_values = []
instance['polling'].disable()  # Disable polling for this example
instance.prepare()
# Run the simulation
for t in times:
    instance['timer'].time = t
    instance.run()
    signal_values.append(instance["attributes"]["signal_value"].internal_value)
    print(f"Time: {t:.2f}s, Signal Value: {signal_values[-1]:.2f}")

Time: -5.00s, Signal Value: -187.00
Time: -4.97s, Signal Value: -185.51
Time: -4.95s, Signal Value: -184.02
Time: -4.92s, Signal Value: -182.55
Time: -4.90s, Signal Value: -181.08
Time: -4.87s, Signal Value: -179.61
Time: -4.85s, Signal Value: -178.15
Time: -4.82s, Signal Value: -176.70
Time: -4.80s, Signal Value: -175.25
Time: -4.77s, Signal Value: -173.81
Time: -4.75s, Signal Value: -172.38
Time: -4.72s, Signal Value: -170.95
Time: -4.70s, Signal Value: -169.53
Time: -4.67s, Signal Value: -168.12
Time: -4.65s, Signal Value: -166.71
Time: -4.62s, Signal Value: -165.30
Time: -4.60s, Signal Value: -163.91
Time: -4.57s, Signal Value: -162.52
Time: -4.55s, Signal Value: -161.13
Time: -4.52s, Signal Value: -159.76
Time: -4.50s, Signal Value: -158.39
Time: -4.47s, Signal Value: -157.02
Time: -4.45s, Signal Value: -155.66
Time: -4.42s, Signal Value: -154.31
Time: -4.40s, Signal Value: -152.96
Time: -4.37s, Signal Value: -151.62
Time: -4.35s, Signal Value: -150.29
Time: -4.32s, Signal Value: 

In [6]:
# Create a Plotly figure
fig = go.Figure()

# Add the interpolated signal trace
fig.add_trace(go.Scatter(
    x=times,
    y=signal_values,
    mode='lines',
    name='Interpolated Signal'
))

# Update the layout of the figure
fig.update_layout(
    title='Interpolated Signal Over Time',
    xaxis_title='Time (s)',
    yaxis_title='Signal Value',
    showlegend=True
)

# Show the figure
fig.show()